## 实验 6：将 class 拆分为 `.hpp + .cpp`

前面的实验把类声明、成员函数定义和使用代码放在同一个 Notebook 中。本实验把 `User` 拆成可复用的接口和实现，并观察预处理、编译与链接各自负责什么。

完成后你应该能够：

- 区分声明、定义和调用；
- 解释 include guard 与 `-I` 头文件搜索路径；
- 解释 `User::` 作用域解析运算符；
- 独立生成 `.o` 文件并链接为可执行文件；
- 根据编译器或链接器报错判断问题发生在哪个阶段。

### 1. 文件结构

```text
02-class-object/
├── 06_header_source.ipynb      # 实验讲解与交互验证
├── 06_header_source_main.cpp   # 使用 User 的翻译单元
├── include/
│   └── user.hpp                # 对外接口：类定义与成员函数声明
└── user.cpp                    # 成员函数定义
```

打开 [user.hpp](include/user.hpp)、[user.cpp](user.cpp) 和 [06_header_source_main.cpp](06_header_source_main.cpp)，对照后面的代码阅读。链接以当前 Notebook 所在目录为基准，因此头文件链接写成 `include/user.hpp`。

### 2. 头文件：公开类的接口

头文件包含完整的 `User` 类定义，其中成员函数只有声明。类定义必须让 `user.cpp` 和主程序都可见，否则编译器不知道对象有哪些成员以及对象需要多少空间。

`#ifndef`、`#define`、`#endif` 组成 include guard，避免同一个翻译单元重复包含该头文件时发生重复定义。

In [ ]:
#include "include/user.hpp"

### 3. 源文件：提供成员函数定义

`user.cpp` 先包含头文件，再提供成员函数的定义。`User::age()` 中的 `::` 是作用域解析运算符，表示这个 `age` 函数属于 `User`。如果省略 `User::`，编译器看到的将是一个无关的普通函数。

下面的交互代码等价于 [user.cpp](user.cpp) 中的实现。请先执行上面的头文件单元格。

In [ ]:
User::User(
    std::string name,
    int age)
    : name_(name),
      age_(age)
{
}

const std::string &
User::name() const
{
    return name_;
}

int User::age() const
{
    return age_;
}

void User::set_age(
    int age)
{
    age_ = age;
}

### 4. 使用类

使用者只需要包含头文件，不需要也不应该 `#include "user.cpp"`。实现文件会在链接阶段以目标文件的形式参与构建。

In [ ]:
#include <iostream>

{
    User user("Bob", 20);

    std::cout << user.name() << ", " << user.age() << '\n';
    user.set_age(21);
    std::cout << user.name() << ", " << user.age() << '\n';
}

### 5. 预处理、编译与链接

`#include` 发生在预处理阶段，本质上是把头文件内容展开到源文件。两个 `.cpp` 会分别编译为两个翻译单元，然后由链接器组合：

```text
                    include/user.hpp
                    /              \
             #include              #include
                 /                    \
            user.cpp       06_header_source_main.cpp
                |                       |
             compile                 compile
                |                       |
             user.o                  main.o
                 \                    /
                  +------ linker -----+
                           |
                    06_header_source
```

在仓库根目录执行以下命令。`-I` 把 `include/` 加入头文件搜索路径，`-c` 表示只编译、不链接：

```bash
mkdir -p /tmp/kotlin-native-study-build
clang++ -std=c++20 -Wall -Wextra -pedantic \
  -I basics/01-cpp/02-class-object/include \
  -c basics/01-cpp/02-class-object/user.cpp \
  -o /tmp/kotlin-native-study-build/user.o
clang++ -std=c++20 -Wall -Wextra -pedantic \
  -I basics/01-cpp/02-class-object/include \
  -c basics/01-cpp/02-class-object/06_header_source_main.cpp \
  -o /tmp/kotlin-native-study-build/main.o
clang++ /tmp/kotlin-native-study-build/main.o \
  /tmp/kotlin-native-study-build/user.o \
  -o /tmp/kotlin-native-study-build/06_header_source
/tmp/kotlin-native-study-build/06_header_source
```

预期输出：

```text
Bob, 20
Bob, 21
```

### 6. 用报错判断故障阶段

- `fatal error: 'user.hpp' file not found`：编译阶段找不到头文件，检查文件位置、链接路径或 `-I`。
- `use of undeclared identifier` / `no member named ...`：声明不可见或声明不匹配，检查头文件。
- `undefined reference` / `Undefined symbols`：声明可见但链接器找不到定义，检查是否遗漏 `user.o` 或函数签名是否一致。
- `duplicate symbol` / `multiple definition`：同一个非 `inline` 定义进入了多个翻译单元，常见原因是把普通函数定义直接写进头文件，或包含了 `.cpp` 文件。

一个很有价值的对照实验：链接时暂时删掉 `user.o`，观察 `User` 构造函数和成员函数的未定义符号错误，然后恢复正确命令。

### 7. 类、对象与 ABI

每个 `User` 对象主要保存非静态数据成员 `name_` 和 `age_`。普通成员函数不会在每个对象中各保存一份；调用成员函数时，当前对象通过隐含的 `this` 指针参与调用。

对象大小还会受到成员类型、对齐、填充、编译器和目标平台影响，因此不要把下面的结果硬编码为固定值。

In [ ]:
std::cout << "sizeof(User) = " << sizeof(User) << '\n';

class LayoutExample
{
private:
    int number_;
    char flag_;
};

std::cout << "sizeof(LayoutExample) = " << sizeof(LayoutExample) << '\n';

### 8. 与 Native SDK 主线的关系

后续 SDK 会沿着同一条构建链继续前进：

```text
公开头文件 + .cpp 实现
          |
       object files
          |
   .a / .so / .dylib
          |
        C ABI
          |
 Kotlin/Native cinterop
```

头文件决定调用方编译时看见的接口；库文件提供链接时需要的实现。C++ ABI 还涉及名称修饰、对象布局和编译器兼容性，因此跨语言 SDK 通常会在 C++ 实现外再提供稳定的 C ABI。

### 9. 练习与验收

1. 修改 `set_age()` 的实现并重新编译，确认只需重编 `user.cpp`，随后重新链接。
2. 去掉 `-I basics/01-cpp/02-class-object/include`，记录头文件搜索错误。
3. 链接时去掉 `user.o`，记录未定义符号错误。
4. 给 `User` 增加一个只读成员函数，在头文件声明、在源文件定义、在主程序调用。
5. 能不看笔记解释：声明用于编译，定义用于链接，include guard 防止同一翻译单元内重复包含。

完成这些练习后，本实验的核心结论是：**头文件公开契约，源文件隐藏实现，每个 `.cpp` 独立编译，链接器负责把跨翻译单元的引用与定义连接起来。**